#### Import Libraries

In [41]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd

#### Read Data

In [55]:
data = pd.read_csv("../data/customData.csv")
data

,cutomerID,Purchase,Total
0,1,Fan,500
1,1,Wires,300
2,1,Bulb,100
3,2,Table,300
4,2,Chair,200
5,2,Door,500
6,2,Window,200
7,3,Fan,500
8,3,Wires,300
9,3,Bulb,100


#### Convert Data from rows to columns
OR simply convert data of customers in to columns of what they have purchases

In [43]:
purchase = (data.groupby(["cutomerID","Purchase"])["Total"]
          .sum().unstack().reset_index().fillna(0)
          .set_index("cutomerID"))

purchase.head()

Purchase,Bulb,Chair,Charger,Computer,Door,Fan,Keyboard,Laptop,Mobile,Mouse,Table,Window,Wires
cutomerID,,,,,,,,,,,,,
1,100.0,0.0,0.0,0.0,0.0,500.0,0.0,0.0,0.0,0.0,0.0,0.0,300.0
2,0.0,200.0,0.0,0.0,500.0,0.0,0.0,0.0,0.0,0.0,300.0,200.0,0.0
3,100.0,0.0,0.0,0.0,0.0,500.0,0.0,0.0,0.0,0.0,0.0,0.0,300.0
4,100.0,0.0,0.0,0.0,0.0,300.0,0.0,0.0,0.0,0.0,0.0,0.0,300.0
5,0.0,200.0,0.0,0.0,500.0,0.0,0.0,0.0,0.0,0.0,300.0,200.0,0.0


#### Do hot encoding to convert data inbetween 0s and 1s not in float numbers

In [44]:
def hot_encode(x):
    if(x<=0):
        return 0
    if(x>=1):
        return 1

In [45]:
purchase = purchase.applymap(hot_encode)
purchase.head()

C:\Users\Sakhi Larik\AppData\Local\Temp\ipykernel_12352\1086531645.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  purchase = purchase.applymap(hot_encode)


Purchase,Bulb,Chair,Charger,Computer,Door,Fan,Keyboard,Laptop,Mobile,Mouse,Table,Window,Wires
cutomerID,,,,,,,,,,,,,
1,1,0,0,0,0,1,0,0,0,0,0,0,1
2,0,1,0,0,1,0,0,0,0,0,1,1,0
3,1,0,0,0,0,1,0,0,0,0,0,0,1
4,1,0,0,0,0,1,0,0,0,0,0,0,1
5,0,1,0,0,1,0,0,0,0,0,1,1,0


Do your machine learning and train the model to check tha data and purchase trend of customers create rules and apriori

1. Find the frequancy of items purchased togather
2. Generate rules that shows the recommondation

In [46]:
freq_item = apriori(purchase,min_support=0.1,use_colnames=True)
freq_item.head()

c:\Users\Sakhi Larik\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:110: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


,support,itemsets
0,0.333333,(Bulb)
1,0.333333,(Chair)
2,0.111111,(Charger)
3,0.222222,(Computer)
4,0.333333,(Door)


In [47]:
rules = association_rules(freq_item,metric="lift",min_threshold=1)

In [48]:
rules.shape

(162, 10)

In [49]:
rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(Bulb),(Fan),0.333333,0.333333,0.333333,1.00,3.00,0.222222,inf,1.000000
1,(Fan),(Bulb),0.333333,0.333333,0.333333,1.00,3.00,0.222222,inf,1.000000
2,(Bulb),(Wires),0.333333,0.444444,0.333333,1.00,2.25,0.185185,inf,0.833333
3,(Wires),(Bulb),0.444444,0.333333,0.333333,0.75,2.25,0.185185,2.666667,1.000000
4,(Door),(Chair),0.333333,0.333333,0.333333,1.00,3.00,0.222222,inf,1.000000


In [50]:
rules[(rules["confidence"] >= 1 ) & (rules["lift"] > 5 ) ]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
28,(Mouse),(Keyboard),0.111111,0.111111,0.111111,1.0,9.0,0.098765,inf,1.0
29,(Keyboard),(Mouse),0.111111,0.111111,0.111111,1.0,9.0,0.098765,inf,1.0
32,(Mobile),(Laptop),0.111111,0.111111,0.111111,1.0,9.0,0.098765,inf,1.0
33,(Laptop),(Mobile),0.111111,0.111111,0.111111,1.0,9.0,0.098765,inf,1.0
66,"(Mouse, Computer)",(Keyboard),0.111111,0.111111,0.111111,1.0,9.0,0.098765,inf,1.0
68,"(Computer, Keyboard)",(Mouse),0.111111,0.111111,0.111111,1.0,9.0,0.098765,inf,1.0
69,(Mouse),"(Computer, Keyboard)",0.111111,0.111111,0.111111,1.0,9.0,0.098765,inf,1.0
71,(Keyboard),"(Mouse, Computer)",0.111111,0.111111,0.111111,1.0,9.0,0.098765,inf,1.0
78,"(Mobile, Computer)",(Laptop),0.111111,0.111111,0.111111,1.0,9.0,0.098765,inf,1.0
80,"(Computer, Laptop)",(Mobile),0.111111,0.111111,0.111111,1.0,9.0,0.098765,inf,1.0


#### Make Recommondations

In [51]:
# What if I want to buy a bulb
rules[rules["antecedents"] == frozenset({'Bulb'})]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(Bulb),(Fan),0.333333,0.333333,0.333333,1.0,3.00,0.222222,inf,1.000000
2,(Bulb),(Wires),0.333333,0.444444,0.333333,1.0,2.25,0.185185,inf,0.833333
45,(Bulb),"(Fan, Wires)",0.333333,0.333333,0.333333,1.0,3.00,0.222222,inf,1.000000


In [52]:
# What if I want to buy a Table
recommond = rules[(rules["antecedents"] == frozenset({'Table'})) & (rules["confidence"] > 0.3)]["consequents"]
recommond

7                    (Chair)
21                (Computer)
23                    (Door)
41                  (Window)
53             (Door, Chair)
65           (Window, Chair)
107           (Window, Door)
133    (Window, Chair, Door)
Name: consequents, dtype: object

In [53]:
# If you want to buy a table you may also want to buy these items
recommond[-1:]

133    (Window, Chair, Door)
Name: consequents, dtype: object